# 03 — Evaluating the customer support agent

> **Applied Labs** · **Domain**: AG · **Difficulty**: Advanced · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/01_customer_support_agent/03_evaluate.ipynb)

**Prerequisites**:
- [02_build.ipynb](./02_build.ipynb) — the full support agent with RAG, tool use, and escalation

**What you will learn**:
- How to build a structured evaluation framework with test cases and metrics
- How to use LLM-as-judge to score response faithfulness and helpfulness
- How to detect hallucinations by checking claims against retrieved context
- How to analyze escalation decisions with confusion matrices and cost analysis
- How to benchmark latency and compute unit economics vs human agents
- How to categorize failures and derive specific improvement actions

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "openai>=1.0.0" "sentence-transformers>=2.2.0" "pandas>=2.0.0" "matplotlib>=3.7.0" "numpy>=1.24.0"

import os
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("Setup complete ✓")

## Section 1 — Evaluation framework

Before we can improve the agent, we need to **measure** it. A robust evaluation framework requires:

| Metric | What it measures | Why it matters |
|--------|-----------------|----------------|
| **Resolution accuracy** | Did the agent correctly answer the question? | Core quality metric |
| **Faithfulness** | Is the response grounded in retrieved context? | Prevents hallucination |
| **Helpfulness** | Did the response actually help the customer? | User satisfaction proxy |
| **Hallucination rate** | % of claims not supported by context | Trust and safety |
| **Escalation precision** | Of escalations, how many were truly necessary? | Avoids wasting human time |
| **Escalation recall** | Of cases needing escalation, how many were caught? | Prevents bad answers |
| **Latency** | Time to generate a response | User experience |

In [ ]:
# Build a test dataset of 30 cases with ground truth

TEST_CASES: list[dict] = [
    # Order status queries
    {"id": "tc-01", "query": "Where is my order #45678?",
     "category": "order_status", "context": "Order #45678 shipped via UPS, tracking 1Z999AA10123456784, estimated delivery Jan 20.",
     "expected_response_contains": ["shipped", "UPS"], "should_escalate": False},
    {"id": "tc-02", "query": "My package hasn't arrived and it's been 10 days",
     "category": "order_status", "context": "Standard shipping takes 3-5 business days. If delayed, allow 2 extra days.",
     "expected_response_contains": ["business days"], "should_escalate": False},
    {"id": "tc-03", "query": "I need to track order #99999",
     "category": "order_status", "context": "No order found with ID #99999.",
     "expected_response_contains": ["not found", "cannot locate"], "should_escalate": False},
    {"id": "tc-04", "query": "Cancel my order before it ships",
     "category": "order_status", "context": "No cancellation policy found in knowledge base.",
     "expected_response_contains": ["specialist", "help"], "should_escalate": True},
    {"id": "tc-05", "query": "When will my express order arrive?",
     "category": "order_status", "context": "Express shipping delivers in 1-2 business days.",
     "expected_response_contains": ["1-2 business days"], "should_escalate": False},
    # Product info queries
    {"id": "tc-06", "query": "What's the battery life on the headphones?",
     "category": "product_info", "context": "Wireless Headphones Pro: 40-hour battery life, ANC, Bluetooth 5.3.",
     "expected_response_contains": ["40-hour", "40 hour"], "should_escalate": False},
    {"id": "tc-07", "query": "Is the speaker waterproof?",
     "category": "product_info", "context": "Portable Speaker Max: IPX7 waterproof rating.",
     "expected_response_contains": ["IPX7", "waterproof"], "should_escalate": False},
    {"id": "tc-08", "query": "Does the charger work with a MacBook Pro?",
     "category": "product_info", "context": "65W GaN USB-C charger, supports PD 3.0, compatible with laptops.",
     "expected_response_contains": ["compatible", "USB-C"], "should_escalate": False},
    {"id": "tc-09", "query": "What switches does the keyboard use?",
     "category": "product_info", "context": "Mechanical Keyboard Elite: Cherry MX Brown switches, hot-swappable.",
     "expected_response_contains": ["Cherry MX Brown"], "should_escalate": False},
    {"id": "tc-10", "query": "How heavy is the wireless mouse?",
     "category": "product_info", "context": "Wireless Mouse Ergo: 4000 DPI, Bluetooth and 2.4GHz. Weight not specified.",
     "expected_response_contains": ["weight", "specification"], "should_escalate": False},
    # Billing queries
    {"id": "tc-11", "query": "I was charged twice for my order",
     "category": "billing", "context": "Duplicate charges often drop in 24 hours. If persistent, refund in 3 business days.",
     "expected_response_contains": ["24 hours", "refund"], "should_escalate": False},
    {"id": "tc-12", "query": "How long until I get my refund?",
     "category": "billing", "context": "Refunds processed in 5-7 business days. Credit cards may take 3-5 additional days.",
     "expected_response_contains": ["5-7 business days"], "should_escalate": False},
    {"id": "tc-13", "query": "I want to return the headphones I bought",
     "category": "billing", "context": "30-day return policy. Must be unused, original packaging. Free return shipping.",
     "expected_response_contains": ["30 days", "30-day"], "should_escalate": False},
    {"id": "tc-14", "query": "My bank flagged your charge as fraud and I'm disputing it",
     "category": "billing", "context": "Duplicate charge resolution: contact support with order number.",
     "expected_response_contains": ["specialist", "help"], "should_escalate": True},
    {"id": "tc-15", "query": "I was overcharged $500 and I want it resolved immediately",
     "category": "billing", "context": "No specific policy for large overcharges.",
     "expected_response_contains": ["specialist", "help"], "should_escalate": True},
    # Technical support queries
    {"id": "tc-16", "query": "My headphones won't connect to Bluetooth",
     "category": "technical_support", "context": "Pairing: enable Bluetooth, hold power 5 sec, select device. Factory reset: power + vol down 10 sec.",
     "expected_response_contains": ["pairing", "Bluetooth"], "should_escalate": False},
    {"id": "tc-17", "query": "How do I update firmware?",
     "category": "technical_support", "context": "Download companion app, connect Bluetooth, Settings > Firmware. Keep 50% charge.",
     "expected_response_contains": ["app", "firmware"], "should_escalate": False},
    {"id": "tc-18", "query": "My device won't charge at all",
     "category": "technical_support", "context": "Try different cable, clean port, ensure 5V/1A charger, try different outlet.",
     "expected_response_contains": ["cable", "charger"], "should_escalate": False},
    {"id": "tc-19", "query": "The speaker makes a buzzing noise at high volume",
     "category": "technical_support", "context": "No specific troubleshooting for buzzing noise in knowledge base.",
     "expected_response_contains": ["specialist", "help"], "should_escalate": True},
    {"id": "tc-20", "query": "My keyboard RGB lights are flickering randomly",
     "category": "technical_support", "context": "No specific troubleshooting for RGB issues in knowledge base.",
     "expected_response_contains": ["specialist", "help"], "should_escalate": True},
    # Complaint queries
    {"id": "tc-21", "query": "This is the worst product I've ever bought",
     "category": "complaint", "context": "30-day return policy for unused items. 90-day for defective.",
     "expected_response_contains": ["sorry", "return"], "should_escalate": False},
    {"id": "tc-22", "query": "I want to speak to a manager right now",
     "category": "complaint", "context": "No manager escalation process in knowledge base.",
     "expected_response_contains": ["specialist", "help", "connect"], "should_escalate": True},
    {"id": "tc-23", "query": "Your company is terrible and I'm posting everywhere",
     "category": "complaint", "context": "No social media response policy in knowledge base.",
     "expected_response_contains": ["sorry", "help"], "should_escalate": True},
    {"id": "tc-24", "query": "Product broke after one day, very disappointed",
     "category": "complaint", "context": "Defective items returnable within 90 days. We cover shipping. Replacement in 2 days.",
     "expected_response_contains": ["90 days", "replacement"], "should_escalate": False},
    {"id": "tc-25", "query": "I've been waiting 3 weeks for support to respond",
     "category": "complaint", "context": "No response time SLA in knowledge base.",
     "expected_response_contains": ["sorry", "apologize"], "should_escalate": True},
    # General queries
    {"id": "tc-26", "query": "How do I create an account?",
     "category": "general", "context": "Visit website, click Sign Up, enter email, create password, verify email.",
     "expected_response_contains": ["sign up", "email"], "should_escalate": False},
    {"id": "tc-27", "query": "Do you ship to Canada?",
     "category": "general", "context": "We ship to 45+ countries. International delivery 7-14 business days.",
     "expected_response_contains": ["international", "ship"], "should_escalate": False},
    {"id": "tc-28", "query": "What does the warranty cover?",
     "category": "general", "context": "1-year limited warranty covering manufacturing defects. Not physical damage.",
     "expected_response_contains": ["1-year", "warranty"], "should_escalate": False},
    {"id": "tc-29", "query": "How do I reset my password?",
     "category": "general", "context": "Click Forgot Password, enter email, receive reset link valid 24 hours.",
     "expected_response_contains": ["forgot password", "reset"], "should_escalate": False},
    {"id": "tc-30", "query": "What payment methods do you accept?",
     "category": "general", "context": "No payment method information in knowledge base.",
     "expected_response_contains": ["specialist", "help"], "should_escalate": True},
]

print(f"Test dataset: {len(TEST_CASES)} cases")
categories = {}
escalation_count = 0
for tc in TEST_CASES:
    categories[tc["category"]] = categories.get(tc["category"], 0) + 1
    if tc["should_escalate"]:
        escalation_count += 1
print(f"Expected escalations: {escalation_count}/{len(TEST_CASES)}")
for cat, count in sorted(categories.items()):
    print(f"  {cat}: {count} cases")

## Section 2 — Response quality (LLM-as-judge)

Human evaluation is expensive and slow. Instead, we use a **separate LLM as an automated judge** to score responses on two dimensions:

- **Faithfulness** (1–5): Is the response grounded in the provided context?
- **Helpfulness** (1–5): Does the response actually address the customer's question?

The judge sees the query, context, and response — but not the ground truth. This simulates how a QA reviewer would evaluate the agent.

In [ ]:
# LLM-as-judge scoring function

JUDGE_PROMPT = """You are evaluating a customer support agent's response.

CUSTOMER QUERY: {query}
CONTEXT PROVIDED TO AGENT: {context}
AGENT RESPONSE: {response}

Score the response on two dimensions (1-5 scale):

FAITHFULNESS: Is the response grounded in the provided context?
  1 = Contains claims contradicting or absent from context
  2 = Mostly unsupported claims
  3 = Mix of supported and unsupported claims
  4 = Mostly grounded, minor extrapolation
  5 = Fully grounded in context

HELPFULNESS: Does the response address the customer's question?
  1 = Completely irrelevant or unhelpful
  2 = Partially relevant but missing key information
  3 = Addresses the question but lacks detail
  4 = Good answer with relevant details
  5 = Excellent, complete answer that fully resolves the query

Respond in JSON format only:
{{"faithfulness": <int>, "helpfulness": <int>, "reasoning": "<brief explanation>"}}
"""


def judge_response(query: str, context: str, response: str,
                   model: str = "gpt-4o-mini") -> dict:
    """Score a response using LLM-as-judge on faithfulness and helpfulness."""
    prompt = JUDGE_PROMPT.format(query=query, context=context, response=response)
    result = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=200,
    )
    try:
        return json.loads(result.choices[0].message.content.strip())
    except json.JSONDecodeError:
        return {"faithfulness": 0, "helpfulness": 0, "reasoning": "Parse error"}


# Generate simulated agent responses for evaluation
def simulate_agent_response(tc: dict) -> str:
    """Generate a simulated agent response for a test case using OpenAI."""
    messages = [
        {"role": "system", "content": (
            "You are a TechGear customer support agent. Be friendly and concise. "
            "Answer using ONLY the provided context. If context is insufficient, "
            "say you'll connect them with a specialist. Keep responses under 100 words."
        )},
        {"role": "user", "content": f"CONTEXT: {tc['context']}\n\nCUSTOMER: {tc['query']}"},
    ]
    result = client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, temperature=0.3, max_tokens=200,
    )
    return result.choices[0].message.content.strip()


# Run evaluation on a subset of test cases
eval_subset = TEST_CASES[:10]
results = []

print("Running LLM-as-judge evaluation on 10 test cases...\n")
for tc in eval_subset:
    response = simulate_agent_response(tc)
    scores = judge_response(tc["query"], tc["context"], response)
    results.append({
        "id": tc["id"],
        "category": tc["category"],
        "faithfulness": scores.get("faithfulness", 0),
        "helpfulness": scores.get("helpfulness", 0),
        "response_preview": response[:80],
    })
    print(f"  {tc['id']}: F={scores.get('faithfulness', 0)} H={scores.get('helpfulness', 0)}")

df_results = pd.DataFrame(results)
print(f"\n{'='*60}")
print(f"Average Faithfulness: {df_results['faithfulness'].mean():.2f}")
print(f"Average Helpfulness:  {df_results['helpfulness'].mean():.2f}")
print(f"\nResults by category:")
print(df_results.groupby("category")[["faithfulness", "helpfulness"]].mean().round(2).to_string())

## Section 3 — Hallucination detection

A response is **hallucinated** if it contains claims not grounded in the retrieved context. We use an NLI-style approach:

1. Extract individual claims from the response
2. For each claim, ask: "Is this supported by the context?"
3. Calculate **grounding rate** = supported claims / total claims

This catches subtle hallucinations that faithfulness scoring might miss — like specific numbers or dates the model invents.

In [ ]:
# Hallucination detection using claim-level grounding checks

HALLUCINATION_PROMPT = """Given the context and a response claim, determine if the claim is SUPPORTED or NOT SUPPORTED by the context.

CONTEXT: {context}
CLAIM: {claim}

Respond with exactly one word: SUPPORTED or NOT_SUPPORTED
"""


def extract_claims(response: str) -> list[str]:
    """Split a response into individual factual claims."""
    sentences = [s.strip() for s in response.replace("\n", " ").split(".") if s.strip()]
    claims = [s for s in sentences if len(s) > 15 and not s.startswith("I ")]
    return claims if claims else [response[:200]]


def check_claim_grounding(claim: str, context: str,
                          model: str = "gpt-4o-mini") -> bool:
    """Check if a single claim is supported by the context."""
    prompt = HALLUCINATION_PROMPT.format(context=context, claim=claim)
    result = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=10,
    )
    answer = result.choices[0].message.content.strip().upper()
    return "SUPPORTED" in answer


def detect_hallucinations(response: str, context: str) -> dict:
    """Analyze a response for hallucinated claims.

    Returns grounding rate and list of unsupported claims.
    """
    claims = extract_claims(response)
    grounded = []
    hallucinated = []
    for claim in claims:
        if check_claim_grounding(claim, context):
            grounded.append(claim)
        else:
            hallucinated.append(claim)
    total = len(claims)
    grounding_rate = len(grounded) / total if total > 0 else 1.0
    return {
        "total_claims": total,
        "grounded": len(grounded),
        "hallucinated": len(hallucinated),
        "grounding_rate": round(grounding_rate, 3),
        "hallucinated_claims": hallucinated,
    }


# Test on a few cases
hallucination_tests = [
    {
        "context": "Wireless Headphones Pro: 40-hour battery, ANC, Bluetooth 5.3. Weight 250g.",
        "response": "The Wireless Headphones Pro has a 40-hour battery life and Bluetooth 5.3 connectivity. "
                    "They also feature active noise cancellation and weigh 250 grams.",
    },
    {
        "context": "Standard shipping takes 3-5 business days. Free on orders over $50.",
        "response": "Your order will arrive in 3-5 business days with free shipping. "
                    "We also offer same-day delivery in major metro areas for an extra $5.",
    },
    {
        "context": "30-day return policy. Must be unused, original packaging. Free return shipping.",
        "response": "You can return items within 30 days. Items must be in original packaging and unused. "
                    "We provide free return shipping labels.",
    },
]

print("Hallucination Detection Results")
print("=" * 60)
for i, test in enumerate(hallucination_tests, 1):
    result = detect_hallucinations(test["response"], test["context"])
    status = "✓ Clean" if result["grounding_rate"] == 1.0 else "⚠ Hallucination detected"
    print(f"\nTest {i}: {status}")
    print(f"  Claims: {result['total_claims']} total, {result['grounded']} grounded, "
          f"{result['hallucinated']} hallucinated")
    print(f"  Grounding rate: {result['grounding_rate']:.1%}")
    if result["hallucinated_claims"]:
        for claim in result["hallucinated_claims"]:
            print(f"  ❌ '{claim[:80]}...'")

## Section 4 — Escalation analysis

Escalation errors have asymmetric costs:
- **False positive** (unnecessary escalation): wastes a human agent's time (~$6/ticket)
- **False negative** (missed escalation): delivers a wrong answer → customer churn, possible refund (~$50+ cost)

We build a confusion matrix and calculate the cost impact of each error type.

In [ ]:
# Confusion matrix for escalation decisions

def build_escalation_matrix(test_cases: list[dict],
                            predictions: list[bool]) -> dict:
    """Build a confusion matrix for escalation decisions.

    Returns counts and rates for TP, FP, TN, FN.
    """
    tp = fp = tn = fn = 0
    for tc, pred in zip(test_cases, predictions):
        actual = tc["should_escalate"]
        if pred and actual:
            tp += 1
        elif pred and not actual:
            fp += 1
        elif not pred and not actual:
            tn += 1
        else:
            fn += 1
    total = len(test_cases)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3),
        "total": total,
    }


# Simulate predictions (realistic: most correct, some errors)
np.random.seed(42)
predictions: list[bool] = []
for tc in TEST_CASES:
    if tc["should_escalate"]:
        predictions.append(np.random.random() < 0.80)  # 80% recall
    else:
        predictions.append(np.random.random() < 0.12)  # 12% false positive rate

matrix = build_escalation_matrix(TEST_CASES, predictions)

# Print confusion matrix
print("Escalation Confusion Matrix")
print("=" * 45)
print(f"                  Predicted")
print(f"                  Escalate    Serve")
print(f"  Actual Esc.     TP={matrix['tp']:<4}     FN={matrix['fn']:<4}")
print(f"  Actual Serve    FP={matrix['fp']:<4}     TN={matrix['tn']:<4}")
print()
print(f"Precision: {matrix['precision']:.3f} (of escalations, how many were necessary)")
print(f"Recall:    {matrix['recall']:.3f} (of necessary escalations, how many were caught)")
print(f"F1 Score:  {matrix['f1']:.3f}")

# Cost analysis
COST_FP = 6.0    # unnecessary escalation: human agent handles easy query
COST_FN = 50.0   # missed escalation: wrong answer → customer churn risk
cost_fp_total = matrix["fp"] * COST_FP
cost_fn_total = matrix["fn"] * COST_FN
print(f"\nCost Analysis (per {matrix['total']} queries):")
print(f"  False positive cost: {matrix['fp']} × ${COST_FP:.0f} = ${cost_fp_total:.0f}")
print(f"  False negative cost: {matrix['fn']} × ${COST_FN:.0f} = ${cost_fn_total:.0f}")
print(f"  Total error cost:    ${cost_fp_total + cost_fn_total:.0f}")
print(f"  Cost per query:      ${(cost_fp_total + cost_fn_total) / matrix['total']:.2f}")

## Section 5 — Latency and cost benchmarking

Production support agents must be fast and cost-effective. We measure:
- **End-to-end latency**: time from query to response
- **Token usage**: input + output tokens per query
- **Cost per query**: based on API pricing
- **Comparison to human agents**: $6–12 per ticket for humans vs our AI cost

In [ ]:
# Latency and cost benchmarking

def benchmark_query(query: str, context: str,
                    model: str = "gpt-4o-mini") -> dict:
    """Benchmark a single query for latency, tokens, and cost."""
    messages = [
        {"role": "system", "content": "You are a customer support agent. Answer concisely using the provided context."},
        {"role": "user", "content": f"CONTEXT: {context}\n\nCUSTOMER: {query}"},
    ]
    start = time.time()
    result = client.chat.completions.create(
        model=model, messages=messages, temperature=0.3, max_tokens=200,
    )
    latency = time.time() - start
    usage = result.usage
    # gpt-4o-mini pricing: $0.15/1M input, $0.60/1M output
    input_cost = (usage.prompt_tokens / 1_000_000) * 0.15
    output_cost = (usage.completion_tokens / 1_000_000) * 0.60
    return {
        "latency_s": round(latency, 3),
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "cost_usd": round(input_cost + output_cost, 6),
    }


# Benchmark 10 queries
benchmark_queries = [
    ("What's the battery life of the headphones?", "Wireless Headphones Pro: 40-hour battery, ANC, Bluetooth 5.3."),
    ("Where is my order #45678?", "Order #45678 shipped via UPS, estimated delivery Jan 20."),
    ("I was charged twice", "Duplicate charges often drop in 24 hours. Refund in 3 business days."),
    ("How do I pair Bluetooth?", "Hold power 5 sec, LED flashes blue, select in Bluetooth settings."),
    ("I want to return my keyboard", "30-day return policy. Must be unused, original packaging. Free return shipping."),
    ("Do you ship to Japan?", "We ship to 45+ countries. International delivery 7-14 business days."),
    ("My speaker won't charge", "Try different cable, clean port, ensure 5V/1A charger."),
    ("What warranty do you offer?", "1-year limited warranty covering manufacturing defects."),
    ("How do I create an account?", "Visit website, click Sign Up, enter email, create password."),
    ("Track my package please", "Tracking number emailed within 24 hours. Check carrier website."),
]

benchmarks = []
print("Running latency benchmarks...\n")
for query, context in benchmark_queries:
    result = benchmark_query(query, context)
    benchmarks.append({"query": query[:40], **result})
    print(f"  {query[:40]:<42} {result['latency_s']:.2f}s  {result['total_tokens']}tok  ${result['cost_usd']:.6f}")

df_bench = pd.DataFrame(benchmarks)

# Summary statistics
print(f"\n{'='*60}")
print("Latency & Cost Summary")
print(f"{'='*60}")
print(f"Latency (mean):     {df_bench['latency_s'].mean():.3f}s")
print(f"Latency (p95):      {df_bench['latency_s'].quantile(0.95):.3f}s")
print(f"Tokens (mean):      {df_bench['total_tokens'].mean():.0f}")
print(f"Cost per query:     ${df_bench['cost_usd'].mean():.6f}")
print(f"Cost per 1K queries: ${df_bench['cost_usd'].mean() * 1000:.3f}")

# Unit economics comparison
ai_cost_per_ticket = df_bench["cost_usd"].mean()
human_cost_low, human_cost_high = 6.0, 12.0
print(f"\nUnit Economics:")
print(f"  AI cost per ticket:     ${ai_cost_per_ticket:.4f}")
print(f"  Human cost per ticket:  ${human_cost_low:.2f} – ${human_cost_high:.2f}")
print(f"  Savings vs human (low): {(1 - ai_cost_per_ticket/human_cost_low)*100:.1f}%")
print(f"  Savings vs human (high):{(1 - ai_cost_per_ticket/human_cost_high)*100:.1f}%")

# Monthly projection
monthly_tickets = 100_000
human_monthly = monthly_tickets * (human_cost_low + human_cost_high) / 2
ai_monthly = monthly_tickets * ai_cost_per_ticket
print(f"\nMonthly projection ({monthly_tickets:,} tickets):")
print(f"  Human cost:    ${human_monthly:>12,.2f}")
print(f"  AI cost:       ${ai_monthly:>12,.2f}")
print(f"  Monthly savings: ${human_monthly - ai_monthly:>12,.2f}")

## Section 6 — Failure analysis

Understanding *why* the agent fails is more valuable than knowing *how often*. We categorize failures into modes and derive specific improvement actions for each.

| Failure mode | Description | Example |
|-------------|-------------|---------|
| **Retrieval miss** | Right intent, wrong documents retrieved | Query about Speaker Max, retrieved Headphones Pro docs |
| **Intent misroute** | Wrong intent classification | Billing complaint classified as general |
| **Insufficient context** | Knowledge base lacks the information | Question about payment methods (not in KB) |
| **Hallucination** | Claims not in context | Invented same-day delivery option |
| **Tone mismatch** | Response tone inappropriate for situation | Cheerful response to angry customer |

In [ ]:
# Failure analysis framework

FAILURE_MODES: list[dict] = [
    {
        "mode": "retrieval_miss",
        "description": "Retrieved documents don't match the query topic",
        "examples": [
            {"query": "How heavy is the wireless mouse?",
             "retrieved": "Wireless Headphones Pro: 250g weight",
             "issue": "Retrieved headphones doc instead of mouse doc"},
            {"query": "Speaker Max dimensions",
             "retrieved": "USB-C charger specs and safety certs",
             "issue": "Wrong product entirely"},
        ],
        "improvement": "Add more product-specific chunks. Improve chunk boundaries to keep product specs together.",
        "priority": "HIGH",
        "estimated_impact": "15-20% of errors",
    },
    {
        "mode": "intent_misroute",
        "description": "Query classified under wrong intent category",
        "examples": [
            {"query": "I want my money back for this broken junk",
             "classified_as": "complaint",
             "should_be": "billing (return/refund)",
             "issue": "Emotional language overrides the actual request"},
            {"query": "Cancel my subscription",
             "classified_as": "general",
             "should_be": "billing",
             "issue": "Subscription not in training examples"},
        ],
        "improvement": "Add more diverse examples to intent centroids. Use multi-label classification.",
        "priority": "HIGH",
        "estimated_impact": "10-15% of errors",
    },
    {
        "mode": "insufficient_context",
        "description": "Knowledge base doesn't contain the needed information",
        "examples": [
            {"query": "What payment methods do you accept?",
             "issue": "No payment method docs in KB"},
            {"query": "Do you have a loyalty program?",
             "issue": "No loyalty program docs in KB"},
        ],
        "improvement": "Audit query logs to find coverage gaps. Add documents for top uncovered topics.",
        "priority": "MEDIUM",
        "estimated_impact": "20-25% of errors",
    },
    {
        "mode": "hallucination",
        "description": "Response contains claims not in the retrieved context",
        "examples": [
            {"query": "Shipping to Canada?",
             "hallucinated": "We offer free shipping to Canada on orders over $75",
             "actual": "No Canada-specific shipping info in context"},
            {"query": "Battery replacement cost?",
             "hallucinated": "Battery replacement costs $29.99",
             "actual": "No pricing info in context"},
        ],
        "improvement": "Strengthen 'only use context' instruction. Add post-generation grounding check.",
        "priority": "CRITICAL",
        "estimated_impact": "5-10% of errors",
    },
    {
        "mode": "tone_mismatch",
        "description": "Response tone doesn't match the customer's emotional state",
        "examples": [
            {"query": "I'm furious! This is the third time this happened!",
             "response_tone": "Cheerful and casual",
             "should_be": "Empathetic and serious"},
        ],
        "improvement": "Add emotion detection. Adjust system prompt dynamically based on customer sentiment.",
        "priority": "MEDIUM",
        "estimated_impact": "5-8% of errors",
    },
]

# Print failure analysis report
print("FAILURE ANALYSIS REPORT")
print("=" * 70)

for fm in FAILURE_MODES:
    print(f"\n{'─' * 70}")
    print(f"Mode: {fm['mode'].upper()}")
    print(f"Priority: {fm['priority']} | Est. impact: {fm['estimated_impact']}")
    print(f"Description: {fm['description']}")
    print(f"Examples:")
    for ex in fm["examples"][:2]:
        print(f"  • Query: '{ex.get('query', 'N/A')}'")
        issue = ex.get("issue", ex.get("hallucinated", ""))
        print(f"    Issue: {issue}")
    print(f"Improvement: {fm['improvement']}")

# Priority matrix
print(f"\n{'='*70}")
print("IMPROVEMENT PRIORITY MATRIX")
print(f"{'='*70}")
print(f"{'Mode':<25} {'Priority':<10} {'Impact':<15} {'Effort'}")
print(f"{'─'*70}")
priority_order = sorted(FAILURE_MODES, key=lambda x: {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2}.get(x["priority"], 3))
efforts = ["Low", "Medium", "Medium", "High", "Medium"]
for fm, effort in zip(priority_order, efforts):
    print(f"{fm['mode']:<25} {fm['priority']:<10} {fm['estimated_impact']:<15} {effort}")

## Exercises

1. **Extend the test dataset**: Add 10 more test cases covering edge cases: multi-language queries, very long messages, queries with typos, and ambiguous intents. Run the LLM-as-judge evaluation and compare scores to the original dataset.

2. **Build an automated regression suite**: Write a function that runs the full evaluation pipeline (faithfulness, hallucination, escalation) and outputs a single pass/fail result based on thresholds (e.g., faithfulness > 3.5, hallucination rate < 10%, escalation recall > 85%). This becomes your CI/CD quality gate.

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | A structured test dataset with ground truth labels is the foundation of systematic evaluation |
| 2 | LLM-as-judge provides scalable quality scoring — but needs careful rubric design to be reliable |
| 3 | Claim-level hallucination detection catches subtle errors that aggregate scores miss |
| 4 | Escalation errors have asymmetric costs — false negatives are 8× more expensive than false positives |
| 5 | AI support costs ~99.9% less per ticket than human agents, with sub-second latency |
| 6 | Failure mode analysis with specific improvements per mode is more actionable than aggregate metrics |

## What's Next

- **Review**: [00_first_principles.ipynb](./00_first_principles.ipynb) — Revisit the first principles now that you've seen the full build-evaluate cycle
- **Related**: [01_architecture.ipynb](./01_architecture.ipynb) — The architecture decisions that drive evaluation outcomes

## References & Further Reading

1. [Zheng et al., "Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena," 2023](https://arxiv.org/abs/2306.05685) — The foundational paper on using LLMs as automated evaluators
2. [Min et al., "FActScore: Fine-grained Atomic Evaluation of Factual Precision in Long Form Text Generation," 2023](https://arxiv.org/abs/2305.14251) — Claim-level factuality evaluation methodology
3. [Gao et al., "RARR: Researching and Revising What Language Models Say, Using Language Models," 2023](https://arxiv.org/abs/2210.08726) — Grounding-based hallucination detection